# 78 - Segmentación con U-Net (Keras) — Base

Plantilla de **U-Net** para segmentación biomédica en datasets pequeños. Si no hay dataset real, se genera uno sintético con formas geométricas.

In [ ]:

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    import numpy as np
    from pathlib import Path
    HAS_TF = True
    print("✅ TensorFlow:", tf.__version__)
except Exception as e:
    HAS_TF = False
    print("⚠️ TensorFlow no disponible:", e)

if HAS_TF:
    # Dataset sintético: imágenes con círculos y cuadrados
    rng = np.random.default_rng(0)
    X = np.zeros((100,128,128,1), dtype="float32")
    Y = np.zeros((100,128,128,1), dtype="float32")
    for i in range(100):
        if rng.random() < 0.5:  # círculo
            rr, cc = np.ogrid[:128,:128]
            mask = (rr-64)**2+(cc-64)**2 < 25**2
            X[i,...,0] = rng.random((128,128))
            Y[i,...,0] = mask.astype("float32")
        else:  # cuadrado
            X[i,...,0] = rng.random((128,128))
            Y[i,32:96,32:96,0] = 1.0

    X_train, Y_train = X[:80], Y[:80]
    X_val, Y_val = X[80:], Y[80:]

    # Definir U-Net pequeña
    def conv_block(x, f):
        x = layers.Conv2D(f,3,padding="same",activation="relu")(x)
        x = layers.Conv2D(f,3,padding="same",activation="relu")(x)
        return x
    def encoder_block(x, f):
        c = conv_block(x,f)
        p = layers.MaxPooling2D()(c)
        return c,p
    def decoder_block(x, s, f):
        us = layers.UpSampling2D()(x)
        concat = layers.Concatenate()([us,s])
        c = conv_block(concat,f)
        return c

    inputs = keras.Input((128,128,1))
    c1,p1 = encoder_block(inputs,16)
    c2,p2 = encoder_block(p1,32)
    c3,p3 = encoder_block(p2,64)
    b = conv_block(p3,128)
    d3 = decoder_block(b,c3,64)
    d2 = decoder_block(d3,c2,32)
    d1 = decoder_block(d2,c1,16)
    outputs = layers.Conv2D(1,1,activation="sigmoid")(d1)

    model = keras.Model(inputs,outputs)
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    hist = model.fit(X_train,Y_train,epochs=3,batch_size=8,validation_data=(X_val,Y_val),verbose=0)
    print("Val acc:", round(hist.history["val_accuracy"][-1],3))
else:
    print("Instala TensorFlow/Keras para ejecutar este notebook.")


## 📝 Ejercicios
1) Aumenta el número de filtros en cada bloque.
2) Cambia el optimizador a AdamW.
3) Prueba con un dataset biomédico real (ej. células, pulmón).